# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) start_async

In [3]:
goal = "When is the next public holiday in Singapore?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

I'll help you find the next public holiday in Singapore. First, let me get the current date and then search for Singapore's public holidays.Now let me search for Singapore's upcoming public holidays for 2025:

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "ccada0ae-67cb-4dae-ae81-621cd1f44d18",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1752389633.1472561,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_017Dc7MkxQXNc6ZKFx3t5Hxz",
                "content": "Current UTC time is 2025-07-13, and the time in Asia/Singapore is 2025-07-13."
            }
        ]
    }
}
{
    "id": "c5759e01-8bcf-4200-a729-2fa0bfb9edca",
    "header": {
        "sender": "ToolUseAgent",
        "recipient": "User",
        "timestamp": 1752389639.4491677,
        "order": 3
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
   

### d) continue_async

In [5]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Based on the official information from Singapore's Ministry of Manpower (MOM), I can tell you the next public holiday in Singapore.

Since today is **July 13, 2025**, the next public holiday in Singapore is:

**National Day - Saturday, August 9, 2025**

This is Singapore's National Day, which commemorates Singapore's independence. Since it falls on a Saturday, it will be observed as a public holiday on that day.

After National Day, the subsequent public holidays for the remainder of 2025 are:
- **Deepavali** - Monday, October 20, 2025
- **Christmas Day** - Thursday, December 25, 2025

So the next public holiday is National Day in about 27 days from today.

### e) interrupt_async

Should be able to interrupt the agent and continue with original task.

In [6]:
resp = await agent.interrupt_async(user_message="Tell me a one paragraph joke.")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but sees nothing except normal bookshelves. He turns back to the librarian, who points to the psychology section and says, "No really, the paranoia books are literally right behind you on that shelf." The man sheepishly walks over, grabs a book, and mutters, "I knew everyone was watching me make a fool of myself." The librarian calls out, "Sir, that's a cookbook." The man looks down at "101 Ways to Prepare Chicken" and sighs, "Great, now even the recipes are plotting against me."

Now, getting back on track - to summarize what we found earlier: The next public holiday in Singapore is **National Day on Saturday, August 9, 2025**, which is approximately 27 days from today (July 13, 2025). Is there anything else you'd like to know about Singapore's public holidays or any other related information?

---
## Scenario 1

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [1]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [2]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.What timezone would you like me to show the current time for? Please provide the timezone in IANA format (e.g., America/New_York, Europe/London, Asia/Tokyo, UTC) or let me know your location and I can determine the appropriate timezone.

Confirm that "continue_async" will not work.

In [3]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


---
## Scenario 2

- LLM Interrupts itself to ask user question
- User responds with answer

In [1]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [2]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.What timezone would you like me to show the current time for? Please provide the timezone in IANA format (e.g., America/New_York, Europe/London, Asia/Tokyo, etc.), or if you're not sure, you can tell me your city or region and I'll help determine the appropriate timezone.

Confirm user cannot continue because LLM is waiting for user input.

In [4]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

Confirm that "continue_async" with message will work when user responds.

In [5]:
resp = await agent.continue_async(user_message="Use SGT")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Perfect! I'll get the current time in Singapore Time (SGT) for you.The current time in Singapore (SGT) is:

**2025-07-13 15:07:20** (3:07:20 PM)

This is Singapore Standard Time, which is UTC+8.

Confirm if its end of response by continuing the agent.

In [6]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
